## MCP reserved method names (JSON-RPC)

On the wire, MCP is JSON-RPC. The strings below are **reserved method names** from the spec — not GitHub-specific. Python `session.list_tools()` is only a wrapper around `"tools/list"`.

A real host (Claude, Codex) speaks these names. GitHub MCP answers the ones its handshake **capabilities** advertised.

### Lifecycle (every connection)

| Method | Direction | What it is |
|--------|-----------|------------|
| `initialize` | client → server | First request: protocol version, clientInfo, client capabilities |
| `notifications/initialized` | client → server | Handshake done; normal requests may follow |
| `ping` | either way | Liveness check |
| `notifications/cancelled` | either way | Cancel an in-flight request |
| `notifications/progress` | either way | Progress for a long call |

### Tools

| Method | What it is |
|--------|------------|
| `tools/list` | Catalog of tools (name, description, JSON Schema) |
| `tools/call` | Run one tool |
| `notifications/tools/list_changed` | Server says the catalog changed; host should list again |

### Resources

| Method | What it is |
|--------|------------|
| `resources/list` | Concrete URIs you can read now |
| `resources/templates/list` | URI patterns with `{placeholders}` |
| `resources/read` | Fetch one URI (file text, HTML UI, …) |
| `resources/subscribe` / `unsubscribe` | Watch a URI for updates |
| `notifications/resources/list_changed` | Catalog changed |
| `notifications/resources/updated` | A subscribed URI changed |

### Prompts

| Method | What it is |
|--------|------------|
| `prompts/list` | Named recipes + argument schema |
| `prompts/get` | Expand a recipe into messages |
| `notifications/prompts/list_changed` | Catalog changed |

### Other (optional capabilities)

| Method | What it is |
|--------|------------|
| `completion/complete` | Autocomplete a prompt arg or URI template placeholder |
| `logging/setLevel` | Ask the server to emit `notifications/message` at a log level |
| `notifications/message` | Server log line (not the handshake) |

Server → client (this notebook does **not** implement these): `sampling/createMessage`, `roots/list`, `elicitation/create`.

The next cell prints the JSON-RPC **method string** on every request, then explores `ping`, `tools/list`, and `completion/complete`.


In [64]:
from dotenv import load_dotenv
import os
import pandas as pd

load_dotenv()

REMOTE_MCP_URL="https://api.githubcopilot.com/mcp"
TOOLSETS="repos, issues, pull_requests, context, users, git"
SERVER_NAME="github"
GITHUB_TOKEN=os.getenv("GITHUB_TOKEN")

In [65]:
def build_github_connection()->dict:
    headers={
        "X-MCP-Toolsets": TOOLSETS,
        "X-MCP-Readonly": "true",
        "Authorization": f"Bearer {GITHUB_TOKEN}"
    }
    return {
        SERVER_NAME: {
            "url": REMOTE_MCP_URL,
            "headers": headers,
            "transport": "http",
            "timeout": 60,
            "sse_read_timeout": 300,
        }
    }

In [66]:
CONNECTIONS=build_github_connection()

In [67]:
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client=MultiServerMCPClient(CONNECTIONS)

In [69]:
def parse_tools(tool_result):
    rows=[]

    for tool in tool_result.tools:
        schema=getattr(tool, "inputSchema", None) or {}

        # if hasattr(tool, "model_dump"):
        #     schema=schema.model_dump()
        
        rows.append({
            "name": tool.name,
            "description": tool.description,
            "input_schema": schema if isinstance(schema, dict) else {}
        })
    
    return sorted(rows, key=lambda x: x["name"])

In [70]:
async def fetch_catalog(client, server_name: str)->dict:
    async with client.session(server_name) as session:
        tools=await session.list_tools()
        tools=parse_tools(tools)
    
    return {
        "tools": tools,
    }

In [71]:
response=await fetch_catalog(mcp_client, SERVER_NAME)

In [72]:
tools_data_frame=pd.DataFrame(
    [
        {
            "tool": t.get("name"),
            "description": t.get("description"),
            "input_schema": t.get("input_schema")
        } for t in response["tools"]
    ]
)

tools_data_frame

,tool,description,input_schema
0,get_commit,Get details for a commit from a GitHub repository,"{'type': 'object', 'properties': {'detail': {'..."
1,get_file_contents,Get the contents of a file or directory from a...,"{'type': 'object', 'properties': {'fields': {'..."
2,get_label,Get a specific label from a repository.,"{'type': 'object', 'properties': {'name': {'ty..."
3,get_latest_release,Get the latest release in a GitHub repository,"{'type': 'object', 'properties': {'owner': {'t..."
4,get_me,Get details of the authenticated GitHub user. ...,"{'type': 'object', 'properties': {}}"
5,get_release_by_tag,Get a specific release by its tag name in a Gi...,"{'type': 'object', 'properties': {'owner': {'t..."
6,get_repository_tree,Get the tree structure (files and directories)...,"{'type': 'object', 'properties': {'owner': {'t..."
7,get_tag,Get details about a specific git tag in a GitH...,"{'type': 'object', 'properties': {'owner': {'t..."
8,get_team_members,Get member usernames of a specific team in an ...,"{'type': 'object', 'properties': {'org': {'typ..."
9,get_teams,Get details of the teams the user is a member ...,"{'type': 'object', 'properties': {'user': {'ty..."
